# Modelos de predicción de sentimiento

En este notebook se entrenan y comparan dos modelos de clasificación binaria para predecir el sentimiento de reseñas hoteleras.

La variable objetivo es `sentiment`, donde:

- `1` representa una reseña positiva.
- `0` representa una reseña negativa.

Los modelos utilizados son Logistic Regression y Random Forest. Como variables predictoras se utilizan el texto completo de la reseña (`review_full_text`) y las variables geográficas `region` y `division`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

import pandas as pd
import matplotlib.pyplot as plt

from src.models.logistic_regression import train_logistic_regression
from src.models.random_forest import train_random_forest, get_feature_importance

In [ ]:
spark = (
    SparkSession.builder
    .appName("HotelReviews-Models-Notebook")
    .getOrCreate()
)

In [ ]:
parquet_path = "../data/processed/hotel_reviews_embeddings.parquet"

df = spark.read.parquet(parquet_path)

df.printSchema()

Aunque el parquet contiene una columna `embedding`, esta no se utiliza en los modelos predictivos. Para esta etapa se utiliza el texto original de la reseña y se transforma mediante TF-IDF dentro del pipeline definido.

In [ ]:
df.select(
    "review_full_text",
    "sentiment",
    "region",
    "division"
).show(5, truncate=False)

In [ ]:
df_model = (
    df
    .select("review_full_text", "sentiment", "region", "division")
    .dropna(subset=["review_full_text", "sentiment", "region", "division"])
)

print("Total de registros para modelado:", df_model.count())

In [ ]:
df_model.groupBy("sentiment").count().orderBy("sentiment").show()

In [ ]:
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

print("Registros de entrenamiento:", train_df.count())
print("Registros de prueba:", test_df.count())

Se utiliza una división 80/20, donde el 80% de los datos se emplea para entrenar los modelos y el 20% restante para evaluarlos. 

**Entrenamiento Logistic Regression:**

In [ ]:
lr_model, lr_predictions, lr_metrics = train_logistic_regression(
    train_df,
    test_df,
    use_region_features=True
)

lr_metrics

In [ ]:
lr_predictions.groupBy("sentiment", "prediction").count().orderBy(
    "sentiment",
    "prediction"
).show()

**Entrenamiento Random Forest:**

In [ ]:
rf_model, rf_predictions, rf_metrics = train_random_forest(
    train_df,
    test_df,
    use_region_features=True
)

rf_metrics

In [ ]:
rf_predictions.groupBy("sentiment", "prediction").count().orderBy(
    "sentiment",
    "prediction"
).show()

**Análisis**

In [ ]:
lr_predictions.select(
    "sentiment",
    "probability"
).show(5, truncate=False)

In [ ]:
rf_predictions.select(
    "sentiment",
    "probability"
).show(5, truncate=False)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

lr_pd = (
    lr_predictions
    .select("sentiment", "probability")
    .toPandas()
)

y_true = lr_pd["sentiment"]

y_score = lr_pd["probability"].apply(
    lambda x: float(x[1])
)

fpr_lr, tpr_lr, _ = roc_curve(y_true, y_score)

auc_lr = auc(fpr_lr, tpr_lr)

In [ ]:
rf_pd = (
    rf_predictions
    .select("sentiment", "probability")
    .toPandas()
)

y_true_rf = rf_pd["sentiment"]

y_score_rf = rf_pd["probability"].apply(
    lambda x: float(x[1])
)

fpr_rf, tpr_rf, _ = roc_curve(
    y_true_rf,
    y_score_rf
)

auc_rf = auc(fpr_rf, tpr_rf)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    fpr_lr,
    tpr_lr,
    label=f"Logistic Regression (AUC={auc_lr:.3f})"
)

plt.plot(
    fpr_rf,
    tpr_rf,
    label=f"Random Forest (AUC={auc_rf:.3f})"
)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curvas ROC")
plt.legend()

plt.show()

In [ ]:
comparison = pd.DataFrame([lr_metrics, rf_metrics])
comparison

In [ ]:
feature_importance = get_feature_importance(rf_model)
feature_importance

## Análisis de resultados

Una vez ejecutados ambos modelos, analizo:

- Cuál modelo obtuvo mejor desempeño general.
- Diferencias entre Accuracy, Precision, Recall, F1-score y AUC-ROC.
- Si el modelo clasifica mejor reseñas positivas o negativas.
- Si el posible desbalance de clases afectó los resultados.
- Si las variables geográficas `region` y `division` aportaron información al modelo.
- Por qué Logistic Regression o Random Forest pudo haber obtenido mejores resultados.